# ECON 4370 / BANA 4373 — Homework 6
## Text and Unstructured Data: From App Reviews to Business Intelligence

**Posted:** Apr. 17 &nbsp;|&nbsp; **Due:** May 1 (11:59 PM, Blackboard Ultra)

**Student name:** TODO

---

### What this homework covers

You will work with **Google Play Store app reviews** — a domain you have never seen
in lecture. The core skills are the same ones from class (text cleaning, TF-IDF,
VADER, OLS), but the context is new: app-store language has its own failure modes,
idioms, and vocabulary that differ meaningfully from movie reviews and restaurant reviews.

By the end you will have:
- Built a full NLP pipeline on a real, unseen corpus
- Regressed VADER sentiment scores on star ratings and interpreted the results
- Identified domain-specific cases where VADER fails
- Connected sentiment measurement error to regression bias

### Two-notebook submission

Submit **two files** to Blackboard Ultra:
1. `HW6_YourName_clean.ipynb` — all outputs **cleared** (Kernel → Restart & Clear Output)
2. `HW6_YourName_executed.ipynb` — fully **run top to bottom**, all outputs visible

### Scoring

| Part | Topic | Points |
|------|-------|--------|
| 1 | Conceptual Questions | 20 |
| 2 | Data Pipeline | 15 |
| 3 | Text Analysis | 20 |
| 4 | Sentiment Scoring and Regression | 30 |
| 5 | Break the Analyzer | 15 |
| **Total** | | **100** |

> **Professionalism deductions (up to −10):** Notebook does not run top → bottom,
> written answers are one sentence with no substance, figures missing from `exports/`,
> or student name not filled in.

---
## Setup — Run this cell first


In [ ]:
# ── Install and import ──────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['datasets', 'nltk', 'scikit-learn', 'statsmodels']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                   capture_output=True)

import os
import re
import numpy as np
import pandas as pd
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
import statsmodels.formula.api as smf
from datasets import load_dataset

for pkg in ['punkt', 'punkt_tab', 'stopwords', 'vader_lexicon']:
    nltk.download(pkg, quiet=True)

for d in ['data_raw', 'data_clean', 'exports']:
    os.makedirs(d, exist_ok=True)

STOP_WORDS = set(stopwords.words('english'))
SIA        = SentimentIntensityAnalyzer()

# SHSU palette
NAVY, TEAL, GOLD, RED = '#1B3A6B', '#007B8A', '#C8A415', '#C0392B'
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})

print('✓ Setup complete')

---
# Part 1 — Conceptual Questions &nbsp; *(20 points)*

Answer each question in the **Markdown cell below it**.
Write in complete sentences. Bullet points are acceptable for lists.
Answers should be substantive: 3–6 sentences per question unless otherwise specified.

## Question 1 &nbsp; *(5 points)*

**TF-IDF vs. raw word frequency.**

a. Explain in your own words what TF-IDF measures and how it differs from a
   simple word frequency count.

b. Suppose the word *"app"* appears 500 times in 1-star reviews and 480 times
   in 5-star reviews. Would TF-IDF flag *"app"* as a distinctive word for either
   group? Why or why not?

c. Name one type of word that raw frequency over-weights and TF-IDF correctly discounts.

**Answer:** TODO


## Question 2 &nbsp; *(5 points)*

**Three VADER failure modes.**

Below are four app-store review snippets. For each one:
- State whether a human would read it as Positive, Negative, or Neutral
- Predict whether VADER would agree or disagree, and why

| # | Review snippet | Human reads | VADER agrees? | Why |
|---|----------------|-------------|---------------|-----|
| 1 | *"Crashes every time I open it."* | | | |
| 2 | *"Works fine I guess."* | | | |
| 3 | *"This update literally killed the app."* | | | |
| 4 | *"Finally fixed! Used to be terrible, now it's actually great."* | | | |

Then name the **general failure pattern** illustrated by snippets 2, 3, and 4
(one sentence each).

**Answer:** TODO

*(Fill in the table above and write the three named failure patterns below.)*


## Question 3 &nbsp; *(5 points)*

**Measurement error and regression bias.**

You want to use VADER compound scores as a regressor in the model:

> `star_rating ~ vader_compound + other_controls`

a. Explain what *classical measurement error* is and what it does to an OLS coefficient.

b. Is the measurement error in VADER compound scores likely to be *classical*
   (random, uncorrelated with the true signal) or *non-classical* (systematic)?
   Justify your answer with reference to Part 5 of this homework.

c. In which direction — toward zero or away from zero — would non-classical
   measurement error in VADER bias the coefficient on `vader_compound`?
   Does your answer depend on the star level? Explain.

**Answer:** TODO


## Question 4 &nbsp; *(5 points)*

**Business scenario: should you use VADER for this?**

A mobile gaming company wants to monitor player satisfaction using VADER sentiment
scores computed on app-store reviews. Their proposed KPI is:

> *Monthly average VADER compound score across all new reviews*

a. List **two statistical concerns** with using this as a KPI
   (think: what could make the number go up or down without any real change in satisfaction?).

b. If the average VADER compound score rises by 0.10 points from January to February,
   give one alternative explanation besides *"players are genuinely happier"*.

c. What is **one additional piece of data** you would want before reporting this
   KPI to the company's product team?

**Answer:** TODO


---
# Part 2 — Data Pipeline &nbsp; *(15 points)*

You will work with the **Google Play Store app reviews** dataset from HuggingFace.
This dataset contains ~25,000 reviews with a `review` (text) column
and a `star` (integer 1–5) column.

> **This is a new domain.** Before writing any code, read at least two reviews
> from the dataset. The vocabulary, phrasing, and failure modes differ from
> the movie and restaurant reviews you saw in lecture.

## Question 5 &nbsp; *(10 points)*

**Load, clean, and tokenize the app reviews.**

Complete the three functions below. Your `clean_text` function must:
- Lowercase the text
- Remove non-alphabetic characters (keep spaces)
- Strip leading/trailing whitespace

Your `tokenize_and_filter` function must:
- Tokenize with `word_tokenize`
- Remove stopwords from `STOP_WORDS`
- Drop tokens shorter than 3 characters

Do **not** copy these functions from the lecture notebook — write them from scratch.

In [ ]:
# ── Load app reviews ────────────────────────────────────────────────────────
print('Loading app reviews...')
ds   = load_dataset('app_reviews', split='train')
df_raw = ds.to_pandas()

# Rename to standard column names used throughout this notebook
df_raw = df_raw.rename(columns={'review': 'text', 'star': 'star'})
df_raw = df_raw[['text', 'star']].dropna().copy()
df_raw['star'] = df_raw['star'].astype(int)

# Stratified sample: 400 reviews per star level → 2,000 total
np.random.seed(42)
frames = []
for star in range(1, 6):
    sub = df_raw[df_raw['star'] == star]
    frames.append(sub.sample(min(400, len(sub))))
df = pd.concat(frames).reset_index(drop=True).copy()

# Read two reviews before writing any analysis code
for star, label in [(1, '1-STAR'), (5, '5-STAR')]:
    ex = df[df['star'] == star].iloc[0]['text']
    print(f'\n{'='*60}')
    print(f'  {label}')
    print(f'{'='*60}')
    print(str(ex)[:400])

df.to_csv('data_raw/app_reviews_raw.csv', index=False)
print(f'\n✓ Raw data saved. Shape: {df.shape}')

In [ ]:
# ── TODO: Define clean_text ─────────────────────────────────────────────────
# Requirements:
#   1. Lowercase
#   2. Remove non-alphabetic characters (keep spaces)
#   3. Collapse multiple spaces to one, strip whitespace

def clean_text(text):
    pass  # YOUR CODE HERE


# ── TODO: Define tokenize_and_filter ────────────────────────────────────────
# Requirements:
#   1. word_tokenize
#   2. Drop stopwords (use STOP_WORDS)
#   3. Drop tokens with fewer than 3 characters

def tokenize_and_filter(text):
    pass  # YOUR CODE HERE


# ── Apply to df ──────────────────────────────────────────────────────────────
df['clean']       = df['text'].apply(clean_text)
df['tokens']      = df['clean'].apply(tokenize_and_filter)
df['token_count'] = df['tokens'].str.len()

print('Pipeline applied.')
print(f'\nOriginal (first 120 chars): {df["text"].iloc[0][:120]}')
print(f'Cleaned  (first 120 chars): {df["clean"].iloc[0][:120]}')
print(f'Tokens   (first 10):        {df["tokens"].iloc[0][:10]}')

## Question 6 &nbsp; *(5 points)*

**Describe the dataset.**

Run the cell below to produce a summary table, then answer the questions in the
Markdown cell that follows.

In [ ]:
# ── Dataset summary ─────────────────────────────────────────────────────────
print(f'Total reviews: {len(df):,}')
print(f'\nReviews per star rating:')
print(df['star'].value_counts().sort_index().to_string())
print(f'\nReview length (words, raw text):')
print(df['text'].str.split().str.len().describe().round(1).to_string())
print(f'\nToken count after cleaning:')
print(df['token_count'].describe().round(1).to_string())

**Q6 written response** *(answer all three)*

a. How does the average review length in this dataset compare to what you saw
   for Yelp or IMDB in lecture? What might explain the difference?

b. What fraction of tokens are removed by the cleaning step (stopwords + short tokens)?
   Is that fraction higher or lower than you expected? Why?

c. Look at the 1-star reviews you printed above. Name **one word or phrase** you saw
   that you would expect VADER to score incorrectly, and explain why.

**Answer:** TODO


---
# Part 3 — Text Analysis &nbsp; *(20 points)*


## Question 7 &nbsp; *(10 points)*

**Word frequency at the rating extremes.**

Compute the top 20 most frequent tokens in 1-star and 5-star reviews separately.
Plot them as two horizontal bar charts (side by side) and save the figure.

Then answer the written questions below.

**Before running:** write your prior below.

✏️ **Prior — write before running:**

List **four words** you expect to appear in the top 20 for **1-star** reviews:  
> TODO

List **four words** you expect to appear in the top 20 for **5-star** reviews:  
> TODO


In [ ]:
# ── TODO: Compute top-20 word frequency for 1-star and 5-star reviews ────────
# Use the 'tokens' column you created in Q5.
# Hint: flatten the list of token lists for each star level, then use Counter.

def top_words(df, star, n=20):
    pass  # YOUR CODE HERE

one_star_freq  = top_words(df, 1)
five_star_freq = top_words(df, 5)

# ── TODO: Plot two side-by-side horizontal bar charts ────────────────────────
# Left:  5-star top words (use TEAL)
# Right: 1-star top words (use RED)
# Save to exports/word_freq_extremes.png

# YOUR CODE HERE

print('1-star top 10:', [w for w, _ in one_star_freq[:10]])
print('5-star top 10:', [w for w, _ in five_star_freq[:10]])

**Q7 written response:**

a. Which of your prior predictions appeared in the top 20? Which did not?

b. Are there words that appear in **both** the 1-star and 5-star top 20?
   What does that tell you about the limitation of raw frequency for
   distinguishing sentiment?

c. Name one word in the 1-star top 20 that a general-purpose sentiment tool
   like VADER might fail to flag as negative. Explain why.

**Answer:** TODO


## Question 8 &nbsp; *(10 points)*

**TF-IDF: what is distinctive, not just frequent.**

Fit a TF-IDF vectorizer on the full corpus. Extract the top 12 most distinctive
words for 1-star and 5-star reviews. Plot them as a side-by-side bar chart
and save the figure.

In [ ]:
# ── TODO: Fit TF-IDF on the full corpus ──────────────────────────────────────
# Use max_df=0.70, min_df=3, max_features=5000

vec = TfidfVectorizer(max_df=0.70, min_df=3, max_features=5000)
# YOUR CODE HERE: fit vec on df['clean']


# ── TODO: Extract top-12 distinctive words for each star level ───────────────
# Join all reviews for a given star level into one string,
# transform it, and rank by TF-IDF score.

def star_tfidf_top(df, star, vec, n=12):
    pass  # YOUR CODE HERE

tfidf_1star = star_tfidf_top(df, 1, vec)
tfidf_5star = star_tfidf_top(df, 5, vec)

# ── TODO: Plot side-by-side horizontal bar charts ────────────────────────────
# Save to exports/tfidf_extremes.png

# YOUR CODE HERE

print('1-star distinctive:', [w for w, _ in tfidf_1star])
print('5-star distinctive:', [w for w, _ in tfidf_5star])

**Q8 written response:**

a. Compare the TF-IDF lists to the raw frequency lists from Q7.
   Identify **two words** that appeared in the raw frequency top 20 but
   dropped out of the TF-IDF top 12. Explain why TF-IDF down-weights them.

b. Do the TF-IDF distinctive words for 1-star reviews match the failure modes
   you noted in Q6c? Are there new failure-mode words that you missed earlier?

c. A colleague argues: *"TF-IDF distinctive words prove that 1-star reviewers
   are more negative — we should use TF-IDF scores directly as a sentiment measure."*
   Give one reason this would be a poor choice compared to VADER.

**Answer:** TODO


---
# Part 4 — Sentiment Scoring and Regression &nbsp; *(30 points)*

> **Reminder:** Run VADER on **raw text**, not cleaned text.
> VADER uses capitalization and punctuation as intensity signals.
> Cleaning destroys them.

## Question 9 &nbsp; *(5 points)*

**VADER scores on app reviews.**

**Before running:** write your predictions below.

✏️ **Prior — write before running:**

What accuracy do you expect VADER to achieve classifying 1-star vs 5-star reviews
as negative vs positive? (Only the extremes — ignore 2, 3, 4 for now.)  
> **Predicted accuracy on extremes:** _______%  &nbsp; **Reasoning:** TODO

Will VADER perform **better or worse** on app reviews compared to IMDB movie reviews?
Justify your prediction in one sentence.  
> TODO


In [ ]:
# ── TODO: Compute VADER compound score for every review ─────────────────────
# Run on raw text (df['text']), cap at 3000 characters for speed.
# Store the compound score in df['vader_compound'].

def vader_compound(text, cap=3000):
    pass  # YOUR CODE HERE

df['vader_compound'] = df['text'].apply(vader_compound)

# ── Summary table: mean VADER compound by star ────────────────────────────────
summary = (df.groupby('star')['vader_compound']
             .agg(['mean', 'median', 'std'])
             .round(3))
summary.columns = ['Mean', 'Median', 'Std Dev']
print('VADER compound by star rating:')
print(summary.to_string())

# ── Accuracy on 1-star vs 5-star only ────────────────────────────────────────
df_extremes = df[df['star'].isin([1, 5])].copy()
df_extremes['vader_pred'] = np.where(df_extremes['vader_compound'] > 0.05,
                                      'positive', 'negative')
df_extremes['true_label'] = np.where(df_extremes['star'] == 5, 'positive', 'negative')
acc = (df_extremes['vader_pred'] == df_extremes['true_label']).mean()
print(f'\nVADER accuracy on 1-star vs 5-star: {acc:.1%}')

**Q9 written response:**

a. How did VADER's actual accuracy compare to your prediction?

b. Look at the mean compound scores by star level.
   Is the relationship monotone (does compound rise consistently from 1 to 5 stars)?
   Where, if anywhere, does it break down?

c. Does VADER perform better or worse on app reviews than on IMDB movie reviews
   (which scored ~85% in lecture)? What does this tell you about domain transfer?

**Answer:** TODO


## Question 10 &nbsp; *(10 points)*

**Baseline OLS: `star ~ vader_compound`.**

**Before running:** write your predictions below.

✏️ **Prior — write before running:**

Expected **sign** of the coefficient on `vader_compound`: &nbsp; TODO  
Expected **R²** (low / medium / high, and why): &nbsp; TODO


In [ ]:
# ── TODO: Fit OLS: star ~ vader_compound ────────────────────────────────────
# Use statsmodels.formula.api (smf.ols)
# Store the fitted model in a variable called `m1`

# YOUR CODE HERE
m1 = None  # replace with your fitted model

print(m1.summary())

# ── Compute residuals ─────────────────────────────────────────────────────────
df['resid_m1'] = m1.resid

In [ ]:
# ── TODO: Scatter plot with OLS line + residual box plot by star ─────────────
# Left panel:  scatter of star (jittered ±0.2) vs vader_compound
#              with OLS line and R² in the legend
# Right panel: box plot of residuals by true star rating
# Save to exports/baseline_ols.png

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# YOUR CODE HERE

plt.tight_layout()
plt.savefig('exports/baseline_ols.png', bbox_inches='tight')
plt.show()

print(f'Coefficient on vader_compound: {m1.params["vader_compound"]:.3f}')
print(f'R²: {m1.rsquared:.3f}')

**Q10 written response:**

a. Interpret the coefficient on `vader_compound` in plain English.
   What does a one-unit change in VADER compound score correspond to
   in terms of predicted star rating?

b. Is the R² higher or lower than you predicted? What fraction of the
   variation in star ratings is *not* explained by VADER compound?

c. Look at the residual box plot. At which star level are the residuals
   largest? Does this match what you saw in lecture, or is the pattern different?

**Answer:** TODO


## Question 11 &nbsp; *(10 points)*

**Extended model: does review length add anything?**

Build two additional OLS specifications and compare them to M1:

| Model | Formula |
|-------|---------|
| M1 | `star ~ vader_compound` (already fitted above) |
| M2 | `star ~ vader_compound + log_token_count` |
| M3 | `star ~ vader_compound + I(star_bin_1) + I(star_bin_5)` |

For M3: create two binary variables — `star_bin_1 = 1` if star == 1, else 0;
`star_bin_5 = 1` if star == 5, else 0. Then fit the same formula.
*(This lets you test whether VADER is systematically wrong at the extremes.)*

**Before running:** predict whether M2 will have a higher R² than M1, and by how much.

✏️ **Prior — write before running:**

Will adding log review length (M2) meaningfully raise R²? Why or why not?  
> TODO

In M3, what sign do you expect on `star_bin_1`? On `star_bin_5`? Why?  
> TODO


In [ ]:
# ── TODO: Create log_token_count and star dummy variables ────────────────────
import numpy as np
df['log_token_count'] = np.log1p(df['token_count'])
df['star_bin_1']      = (df['star'] == 1).astype(int)
df['star_bin_5']      = (df['star'] == 5).astype(int)

# ── TODO: Fit M2 and M3 ───────────────────────────────────────────────────────
# Store as m2 and m3

# YOUR CODE HERE
m2 = None  # replace
m3 = None  # replace

# ── TODO: Print a summary_col comparison table ────────────────────────────────
from statsmodels.iolib.summary2 import summary_col

# YOUR CODE HERE

In [ ]:
# ── TODO: Bar chart comparing R² and Adj. R² across M1, M2, M3 ──────────────
# Save to exports/model_comparison.png

# YOUR CODE HERE

print(f'M1  R²: {m1.rsquared:.3f}')
print(f'M2  R²: {m2.rsquared:.3f}   Delta vs M1: {m2.rsquared - m1.rsquared:+.4f}')
print(f'M3  R²: {m3.rsquared:.3f}   Delta vs M1: {m3.rsquared - m1.rsquared:+.4f}')

**Q11 written response:**

a. Did adding log review length (M2) meaningfully raise R²?
   Does this match your prior? What does it imply about the relationship
   between how much someone writes and how they rate the app?

b. Interpret the coefficients on `star_bin_1` and `star_bin_5` in M3.
   What do they tell you about *where* VADER's predictions are systematically off?

c. Based on M3, is VADER more wrong at the positive extreme or the negative extreme
   of the rating scale for this corpus?
   How does this compare to what you found with Yelp data in lecture?

**Answer:** TODO


## Question 12 &nbsp; *(5 points)*

**Export the final dataset and summarize the regression arc.**


In [ ]:
# ── Save clean dataset ───────────────────────────────────────────────────────
df[['text', 'clean', 'star', 'vader_compound',
    'token_count', 'log_token_count', 'resid_m1']].to_csv(
    'data_clean/app_reviews_clean.csv', index=False)
print('✓ data_clean/app_reviews_clean.csv saved')

**Q12 written response — the full regression arc (4–6 sentences):**

Starting from the baseline VADER compound score and ending with M3,
write a brief narrative that:

1. States what the baseline OLS tells you (coefficient, R²)
2. Explains what M2 showed about review length
3. Explains what M3 revealed about systematic bias at the star extremes
4. Concludes with one sentence on whether you would trust VADER compound
   as a measure of app satisfaction in a real business analysis, and why

**Answer:** TODO


---
# Part 5 — Break the Analyzer &nbsp; *(15 points)*

In lecture you saw VADER fail on movie-review sarcasm and on electronics-domain idioms.
App-store reviews have their own failure modes.

> **Read before you code.** Look at the 1-star and 2-star reviews in `df` before
> starting Q13. Scroll through at least 10 reviews. You are looking for phrases
> that a human would read as clearly negative but that VADER might mis-score.

## Question 13 &nbsp; *(8 points)*

**Build an app-review failure table.**

Find **5 phrases** from the app-review corpus that you expect VADER to mis-score.
At least 3 of the 5 must come from **actual reviews in the dataset** (not invented).

Run each phrase through VADER and fill the table below.
For each phrase, name the **failure pattern** from the following list:

- *Domain idiom* — phrase has special negative meaning in app reviews but not in general English
- *Expectation language* — negative meaning carried by implication, not keywords
- *Negation scope* — VADER mis-handles a negated negative
- *Sarcasm/irony* — positive surface wording with negative intent
- *Version complaint* — technical update language with clear negative valence

In [ ]:
# ── TODO: Define your 5 phrases ──────────────────────────────────────────────
# Format: (phrase_string, 'Positive'/'Negative'/'Neutral', 'Failure pattern name')
# At least 3 phrases must come from actual reviews in df.

phrases = [
    # ('phrase here', 'Human reads: Negative', 'Failure pattern'),
    # YOUR CODE HERE — add 5 tuples
]

# ── Run VADER on each phrase and print results ────────────────────────────────
print(f"{'PHRASE':<48}  {'SCORE':>7}  VADER CALL    HUMAN READS")
print('─' * 100)
for phrase, human_read, pattern in phrases:
    score = SIA.polarity_scores(phrase)['compound']
    call  = 'POSITIVE' if score > 0.05 else ('NEGATIVE' if score < -0.05 else 'NEUTRAL')
    correct = ((call == 'POSITIVE' and 'Positive' in human_read) or
               (call == 'NEGATIVE' and 'Negative' in human_read))
    flag = '✓' if correct else '✗ WRONG'
    print(f'{phrase[:47]:<48}  {score:>+7.3f}  {call:<12}  {flag} | {human_read}')
    print(f'   Pattern: {pattern}\n')

**Q13 written response:**

a. Which failure pattern appeared most often in your 5 phrases?
   Is that pattern specific to app reviews, or would it appear equally often
   in movie or restaurant reviews?

b. Pick your **most interesting** mis-scored phrase and explain in 2–3 sentences
   exactly *why* VADER gets it wrong at the token level
   (i.e., which specific words are misleading VADER and in which direction).

**Answer:** TODO


## Question 14 &nbsp; *(7 points)*

**Residual analysis: where is VADER most wrong in the corpus?**

Use the M1 residuals (`df['resid_m1']`) to find and visualize where VADER
makes its largest prediction errors.

**Before running:** write your prediction below.

✏️ **Prior — write before running:**

At which star level do you expect the **largest mean absolute residual**?  
> TODO

Do you expect more *false optimists* (VADER too positive) or
*false pessimists* (VADER too negative) in this corpus, and why?  
> TODO


In [ ]:
# ── TODO: Compute mean absolute residual by star level ───────────────────────
df['abs_resid_m1'] = df['resid_m1'].abs()

# YOUR CODE HERE: compute mae_by_star (grouped mean of abs_resid_m1)


# ── TODO: Identify false optimists and false pessimists ───────────────────────
# false optimist:  resid_m1 < -1  (model predicted too high → VADER too positive)
# false pessimist: resid_m1 >  1  (model predicted too low  → VADER too negative)

# YOUR CODE HERE


# ── TODO: Print the 3 worst false optimists and 3 worst false pessimists ──────
# For each: print actual star, vader_compound, residual, and first 250 chars of text

# YOUR CODE HERE

In [ ]:
# ── TODO: Two-panel figure ───────────────────────────────────────────────────
# Left:  bar chart of mean absolute residual by star (color-code the bars)
# Right: scatter of vader_compound vs star (jittered), points colored by
#        error direction (red = false optimist, teal = false pessimist)
#        with M1 OLS line overlaid
# Save to exports/residual_deepdive.png

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# YOUR CODE HERE

plt.tight_layout()
plt.savefig('exports/residual_deepdive.png', bbox_inches='tight')
plt.show()

n_opt = (df['resid_m1'] < -1).sum()
n_pes = (df['resid_m1'] >  1).sum()
print(f'False optimists  (residual < −1): {n_opt} reviews')
print(f'False pessimists (residual >  1): {n_pes} reviews')
print(f'\nMean absolute residual by star:')
print(mae_by_star.round(3).to_string())

**Q14 written response:**

a. Which star level has the largest mean absolute residual?
   Does this match the pattern you saw in lecture with Yelp reviews,
   or is the app-review pattern different? Suggest a reason for any difference.

b. Read the 3 worst false-optimist reviews you printed.
   Do they contain any of the failure patterns you identified in Q13?
   Name the specific words or phrases that fooled VADER.

c. Are there more false optimists or false pessimists?
   What does the asymmetry (if any) imply about the direction of bias
   in the M1 regression coefficient?

**Answer:** TODO


---
## Final Export


In [ ]:
# ── Final export ─────────────────────────────────────────────────────────────
df[['text', 'clean', 'star', 'vader_compound',
    'token_count', 'log_token_count', 'resid_m1',
    'star_bin_1', 'star_bin_5']].to_csv(
    'data_clean/app_reviews_clean.csv', index=False)

print('✓ data_clean/app_reviews_raw.csv')
print('✓ data_clean/app_reviews_clean.csv')
print('\nExports folder:')
for f in sorted(os.listdir('exports')):
    print(' ', f)

---
## Submission Checklist

Before uploading to Blackboard, confirm **every item** below:

**Part 1 — Conceptual**
- [ ] Q1: TF-IDF explanation + both sub-questions answered
- [ ] Q2: Table completed for all 4 snippets; 3 failure patterns named
- [ ] Q3: Measurement error mechanism explained, direction of bias stated
- [ ] Q4: Two KPI concerns listed; alternative explanation given; additional data named

**Part 2 — Data Pipeline**
- [ ] Q5: `clean_text` and `tokenize_and_filter` written from scratch (not copied from lecture)
- [ ] Q5: `data_raw/app_reviews_raw.csv` saved
- [ ] Q6: All three written sub-questions answered

**Part 3 — Text Analysis**
- [ ] Q7: Prior written before running; `exports/word_freq_extremes.png` saved
- [ ] Q7: All three written sub-questions answered
- [ ] Q8: `exports/tfidf_extremes.png` saved; all three written sub-questions answered

**Part 4 — Sentiment & Regression**
- [ ] Q9: Prior written before running; accuracy reported and interpreted
- [ ] Q10: `m1` fitted; `exports/baseline_ols.png` saved; coefficient interpreted
- [ ] Q11: Prior written; `m2` and `m3` fitted; `exports/model_comparison.png` saved
- [ ] Q12: `data_clean/app_reviews_clean.csv` saved; regression arc narrative written

**Part 5 — Break the Analyzer**
- [ ] Q13: 5 phrases collected (≥3 from actual reviews); VADER table printed; pattern named
- [ ] Q14: Prior written; `exports/residual_deepdive.png` saved; all three sub-questions answered

**Final**
- [ ] Notebook runs **top → bottom without errors**
- [ ] All `TODO` placeholders replaced with real answers
- [ ] **Student name** filled in at the top
- [ ] Two files uploaded: `HW6_YourName_clean.ipynb` and `HW6_YourName_executed.ipynb`

---
### Further Reading (Optional)
- **Hutto & Gilbert (2014)**, *VADER: A Parsimonious Rule-Based Model for Sentiment Analysis* — the original paper
- **Jurafsky & Martin**, *Speech and Language Processing* (3rd ed., free online), Ch. 4 (Naïve Bayes & Sentiment)
- **Cunningham (2021)**, *Causal Inference: The Mixtape*, Ch. 2 (Potential Outcomes) — for the measurement error framing in Q3


---
## Grading Rubric

| Section | Points | What we look for |
|---------|--------|------------------|
| **Part 1: Conceptual** | 20 | Accurate explanations of TF-IDF (Q1), VADER failure modes with correct table entries (Q2), measurement error mechanism and direction (Q3), two distinct KPI concerns and a plausible alternative explanation (Q4) |
| **Part 2: Data Pipeline** | 15 | `clean_text` and `tokenize_and_filter` implemented correctly from scratch; pipeline applied and verified; written Q6 responses are specific to this dataset |
| **Part 3: Text Analysis** | 20 | Word frequency bar charts correct and properly labeled (Q7); TF-IDF vectorizer fitted and star-level extraction correct (Q8); written responses compare raw frequency vs. TF-IDF with a concrete example |
| **Part 4: Sentiment & Regression** | 30 | VADER scores computed on raw text (Q9); M1 fitted and scatter/residual plot saved (Q10); M2 and M3 fitted with summary_col table and R² chart (Q11); regression arc narrative is specific and substantive (Q12) |
| **Part 5: Break the Analyzer** | 15 | ≥3 phrases drawn from actual reviews; VADER table printed with correct failure pattern labels (Q13); residual MAE bar chart + scatter saved; false optimist reviews read and connected to Q13 failure patterns (Q14) |
| **Total** | **100** | |

> **Professionalism deductions (up to −10):**
> Notebook does not run top → bottom,
> written answers are one sentence with no substance,
> figures missing from `exports/`,
> or student name not filled in at the top.